# Day 1: Secret Entrance

[*Advent of Code 2025 day 1*](https://adventofcode.com/2025/day/1) and [*solution megathread*](https://redd.it/1pb3y8p)

[![nbviewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.jupyter.org/github/UncleCJ/advent-of-code/blob/cj/2025/01/code.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/UncleCJ/advent-of-code/cj?filepath=2025%2F01%2Fcode.ipynb)

In [1]:
from IPython.display import HTML
import sys
sys.path.append('../../')


# %load_ext nb_mypy
# %nb_mypy On

In [2]:
import common


downloaded = common.refresh()
%store downloaded >downloaded

# %load_ext pycodestyle_magic
# %pycodestyle_on

Writing 'downloaded' (dict) to file 'downloaded'.


In [3]:
from IPython.display import HTML

HTML(downloaded['part1'])

In [4]:
part1_example_input = '''L68
L30
R48
L5
R60
L55
L1
L99
R14
L82'''

inputdata = downloaded['input']

In [5]:
def parse_input(lines):
    return [(line[0], int(line[1:])) for line in lines]

# parsed_input = parse_input(part1_example_input.splitlines())
parsed_input = parse_input(inputdata.splitlines())

First, the naive solution:

In [6]:
position = 50
zero_count = 0

for direction, distance in parsed_input:
    if direction == 'L':
        position = (position - distance) % 100
    else:
        position = (position + distance) % 100
    
    if position == 0:
        zero_count += 1

print(zero_count)

1048


Then, backdated from part 2, the beginning of a general solution. I intend to implement things nicer as filters, reductions, something, this year, so that's the reason for the state and rotation tuples:

In [7]:
from typing import Tuple


def rotate_dial(state: Tuple[int, int], count_all_zeros: bool, rotation: Tuple[str, int] ) -> Tuple[int, int]:
    position0, zero_count = state
    direction, distance = rotation
    match (direction):
        case 'L':
            position = position0 - distance
        case 'R':
            position = position0 + distance
    if (count_all_zeros):
        zero_count += abs(position) // 100
        # Count going negative once, assuming we didn't start at zero
        if (position < 0 and position0 != 0):
            zero_count += 1
        # If we landed at 0 and have moved at least one turn,
        # it must be discounted as it will be counted below
        if (position % 100 == 0 and abs(position) >= 100):
            zero_count -= 1
    position = position % 100
    if (position == 0):
        zero_count += 1
    # print(f'{state=}, {rotation=} => {position=}, {zero_count=}')
    return (position, zero_count)

Usually I'm horrible about thinking in terms of tests, but here it helped with some basic unit tests:

In [8]:
assert(rotate_dial((10, 0), True, ('R', 90)) == (0, 1))
assert(rotate_dial((0, 0), True, ('R', 100)) == (0, 1))
assert(rotate_dial((10, 0), True, ('R', 100)) == (10, 1))
assert(rotate_dial((0, 0), True, ('L', 100)) == (0, 1))
assert(rotate_dial((0, 0), True, ('L', 200)) == (0, 2))
assert(rotate_dial((0, 0), True, ('L', 110)) == (90, 1))

In [9]:
position, zero_count = (50, 0)

for input_step in parsed_input:
    position, zero_count = rotate_dial((position, zero_count), False, input_step)

print(zero_count)

1048


In [10]:
HTML(downloaded['part1_footer'])

In [11]:
HTML(downloaded['part2'])

In [12]:
position, zero_count = (50, 0)

for input_step in parsed_input:
    position, zero_count = rotate_dial((position, zero_count), True, input_step)

print(zero_count)

6498


In [13]:
HTML(downloaded['part2_footer'])